# STAD CSV → RDF:

This script generates rdf files in turtle syntax from csv files using RDFLib.Place this notebook in the same folder as "Hourly Temp data" folder containing downloaded hourly temperature readings or alternatively, edit the path in the setup cell to point to the data. Install dependencies with `%pip install pandas rdflib` if needed, then run from top to bottom.

## Setup and paths



In [ ]:
import calendar
import datetime
import hashlib
import json
import re
from pathlib import Path
from statistics import mean

import pandas as pd
from rdflib import BNode, Graph, Literal, Namespace, OWL, RDF, RDFS, URIRef, XSD
from rdflib.collection import Collection

# Edit paths to match your folder structure.
DATA_DIR = Path.cwd()                       
#ONTOLOGY_DIR = DATA_DIR                     # contains STAD.ttl and STAD-MLS.ttl
SAMPLE_DIR = DATA_DIR / "Sample data"
OUTPUT_DIR = DATA_DIR / "sample data ontology"
#VERIFY_SERIALIZATION = True
#EXPORT_STATE_BOUNDARY = False

# Namespaces used in ontology
ex = Namespace("http://stad.spatialai.org/example/")
stad = Namespace("http://purl.org/spatialai/stad/v2/core#")
stad_mls = Namespace("http://purl.org/spatialai/stad/v2/mls#")
qudt = Namespace("http://qudt.org/schema/qudt/")
unit = Namespace("http://qudt.org/vocab/unit/")
time = Namespace("http://www.w3.org/2006/time#")
geo = Namespace("http://www.opengis.net/ont/geosparql#")

STATIONS = {2043: ex.MascomaRiverWeatherStation, 2069: ex.HubbardBrookWeatherStation}
TEMP_COLUMN = "TOBS.I-1 (degC) "
FIXED_OFFSET = "-05:00"              
RUN_SUMMARY = []
DEFINED_DATAPOINTS = set()
REFERENCED_DATAPOINTS = set()

def new_graph():
    graph = Graph()
    for prefix, namespace in [("ex", ex), ("stad", stad), ("stad-mls", stad_mls),
                              ("qudt", qudt), ("unit", unit), ("time", time),
                              ("geo", geo), ("rdf", RDF), ("rdfs", RDFS),
                              ("owl", OWL), ("xsd", XSD)]:
        graph.bind(prefix, namespace)
    return graph


## RDFLib helpers for values, intervals, datasets, and provenance



In [ ]:
def stable_blank(owner, role):
    """Stable identifiers merge shared metadata when the generated graphs are combined."""
    digest = hashlib.sha256(f"{owner}|{role}".encode()).hexdigest()
    return BNode("stad" + digest)

def as_date(value):
    if isinstance(value, str):
        return datetime.datetime.strptime(value, "%m/%d/%Y").date()
    return pd.Timestamp(value).date()

def observation_iri(row):
    site = int(row["Site Id"])
    day = as_date(row["Date"]).strftime("%Y%m%d")
    hour = datetime.datetime.strptime(str(row["Time"]), "%H:%M").strftime("%H")
    return ex[f"{site}_{day}_{hour}"]

def stamp(day, clock="00:00:00"):
    return f"{day}T{clock}{FIXED_OFFSET}"

def add_interval(graph, iri, beginning, ending, coverage=False):
    if datetime.datetime.fromisoformat(beginning) > datetime.datetime.fromisoformat(ending):
        raise ValueError(f"Reversed interval: {iri}")
    graph.add((iri, RDF.type, time.ProperInterval))
    if coverage:
        graph.add((iri, RDF.type, stad.TemporalCoverage))
    for relation, label, value in [(time.hasBeginning, "beginning", beginning),
                                   (time.hasEnd, "end", ending)]:
        instant = stable_blank(iri, label)
        graph.add((iri, relation, instant))
        graph.add((instant, RDF.type, time.Instant))
        graph.add((instant, time.inXSDDateTimeStamp,
                   Literal(value, datatype=XSD.dateTimeStamp, normalize=False)))
    return iri

def add_value(graph, owner, value, decimals=2):
    node = stable_blank(owner, "temperature-value")
    numeric = float(value)
    if decimals is not None:
        numeric = round(numeric, decimals)
    graph.add((owner, stad.hasQuantitativeValue, node))
    graph.add((node, RDF.type, stad.QuantitativeValue))
    graph.add((node, RDF.type, qudt.QuantityValue))
    graph.add((node, qudt.numericValue, Literal(numeric, datatype=XSD.float)))
    graph.add((node, qudt.unit, unit.DEG_C))
    return node

def add_description(graph, description, temporal_unit=time.unitHour):
    graph.add((description, RDF.type, stad.DatasetDescription))
    resolution = stable_blank(description, "temporal-resolution")
    duration = stable_blank(description, "temporal-duration")
    graph.add((description, stad.hasTemporalResolution, resolution))
    graph.add((resolution, RDF.type, stad.TemporalResolution))
    graph.add((resolution, stad.hasQualityKind, stad.temporalResolutionQualityKind))
    graph.add((resolution, stad.hasQuantitativeValue, duration))
    graph.add((duration, RDF.type, time.Duration))
    graph.add((duration, RDF.type, stad.QuantitativeValue))
    graph.add((duration, time.numericDuration, Literal("1.0", datatype=XSD.decimal)))
    graph.add((duration, time.unitType, temporal_unit))
    return description

def add_dataset(graph, dataset, members, description=None, temporal_unit=time.unitHour):
    description = description or ex[str(dataset).removeprefix(str(ex)) + "Description"]
    graph.add((dataset, RDF.type, stad.Dataset))
    graph.add((dataset, stad.hasDatasetDescription, description))
    # The shared SCAN description comes from the normalized sample core.
    if description != ex.SCAN_BaseDataDescription:
        add_description(graph, description, temporal_unit)
    for member in members:
        graph.add((dataset, stad.containsDatapoint, member))
    return dataset

METHODS = {
    "mean": (ex.PandasArithmeticMeanExecution, ex.ArithmeticMeanImplementationInPandas,
             stad.ArithmeticMeanCalculation),
    "min": (ex.PandasArithmeticMinimumExecution, ex.ArithmeticMinimumImplementationInPandas,
            stad.MinimumCalculation),
    "max": (ex.PandasArithmeticMaximumExecution, ex.ArithmeticMaximumImplementationInPandas,
            stad.MaximumCalculation),
}

def add_execution(graph, datapoint, execution, dataset, method):
    execution_class, implementation, algorithm = METHODS[method]
    graph.add((datapoint, stad.generatedBy, execution))
    graph.add((datapoint, stad.hasTransformationKind, algorithm))
    graph.add((execution, RDF.type, execution_class))
    graph.add((execution, RDF.type, stad_mls.AlgorithmExecution))
    graph.add((execution, stad_mls.hasInput, dataset))
    graph.add((execution, stad_mls.executes, implementation))
    graph.add((execution, stad_mls.realizes, algorithm))
    # Current pandas Series reductions use axis=0, skipna=True, numeric_only=False.
    for setting in (ex.AxisParameterSetting, ex.SkipNaParameterSetting, ex.NumericOnlyParameterSetting):
        graph.add((execution, stad_mls.hasInput, setting))

def add_convex_period(graph, period, extent, beginning, ending):
    graph.add((period, RDF.type, stad.ConvexAggregationPeriod))
    graph.add((period, stad.hasExtent, extent))
    graph.add((period, time.hasPart, extent))
    add_interval(graph, extent, beginning, ending)
    return period

def add_nonconvex_period(graph, period, extent, parts):
    parts = list(parts)
    if not parts:
        raise ValueError("A non-convex period needs temporal parts")
    graph.add((period, RDF.type, stad.NonConvexAggregationPeriod))
    graph.add((period, stad.hasExtent, extent))
    add_interval(graph, extent, min(a for _, a, _, _ in parts), max(b for _, _, b, _ in parts))
    for part, beginning, ending, part_class in parts:
        add_interval(graph, part, beginning, ending)
        graph.add((part, RDF.type, part_class))
        graph.add((period, time.hasPart, part))
        graph.add((part, time.isPartOf, period))
        graph.add((part, time.intervalIn, ex["TimeInterval_2013-2022"]))
    return period

def add_aggregate(graph, datapoint, value, quality, execution, dataset, coverage,
                  period, method="mean", site_id=None, spatial=False):
    aggregate_class = stad.SpatioTemporalAggregateDatapoint if spatial else stad.TemporalAggregateDatapoint
    graph.add((datapoint, RDF.type, aggregate_class))
    graph.add((datapoint, RDF.type, stad.QuantitativeDatapoint))
    graph.add((datapoint, stad.hasQualityKind, quality))
    graph.add((datapoint, stad.hasTemporalCoverage, coverage))
    graph.add((datapoint, stad.hasAggregationPeriod, period))
    graph.add((datapoint, stad.hasBaseDataset, dataset))
    if spatial:
        graph.add((datapoint, stad.hasSpatialCoverage, ex.NewHampshire))
    if site_id is not None:
        graph.add((datapoint, ex.measuredAt, STATIONS[int(site_id)]))
    add_value(graph, datapoint, value)
    add_execution(graph, datapoint, execution, dataset, method)
    return graph


def save_graph(graph, filename):
    path = OUTPUT_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)

    graph.serialize(
        destination=str(path),
        format="turtle",
        encoding="utf-8"
    )

    print(f"{filename}: {len(graph):,} triples saved", flush=True)

## Load and validate CSV inputs once



In [ ]:
folder = DATA_DIR / "Hourly Temp data"
files = sorted(folder.glob("*.csv"))
if not files:
    raise FileNotFoundError(f"No CSV files found: {folder}")
frames = {}
for filename in files:
    df = pd.read_csv(filename)
    required = {"Site Id", "Date", "Time", TEMP_COLUMN}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns in {filename}: {required - set(df.columns)}")
    if len(df) == 0 or df["Site Id"].nunique() != 1:
        raise ValueError(f"Expected a non-empty single-station file: {filename}")
    if int(df["Site Id"].iloc[0]) not in STATIONS:
        raise ValueError(f"Add station metadata for {filename}")
    df[TEMP_COLUMN] = pd.to_numeric(df[TEMP_COLUMN], errors="raise")
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
    if not all(datetime.datetime.strptime(str(t), "%H:%M").minute == 0 for t in df["Time"]):
        raise ValueError("Hourly observation identifiers require timestamps at whole hours")
    keys = df["Date"].dt.strftime("%Y%m%d") + "_" + df["Time"].astype(str)
    if keys.duplicated().any():
        raise ValueError(f"Duplicate hourly identifiers: {filename}")
    frames[filename.stem] = df

def year_frame(year):
    selected = [df for name, df in frames.items() if int(name.split("=")[-1]) == year]
    if not selected:
        raise ValueError(f"No input files for {year}")
    return pd.concat(selected, ignore_index=True)

def summer_frame(year):
    df = year_frame(year)
    return df[(df["Date"] >= pd.Timestamp(year=year, month=6, day=1)) &
              (df["Date"] <= pd.Timestamp(year=year, month=8, day=31))].copy()

def annual_winter_frame(year):
    previous, current = year_frame(year-1), year_frame(year)
    return pd.concat([previous[previous["Date"].dt.month == 12],
                      current[current["Date"].dt.month.isin([1, 2])]], ignore_index=True)


## Hourly observations

Transforms hourly observations to to rdf.

In [ ]:
def create_ontology_observedTemp(df):
    graph = new_graph()
    for _, row in df.iterrows():
        datapoint = observation_iri(row)
        graph.add((datapoint, RDF.type, stad.ObservationalDatapoint))
        graph.add((datapoint, RDF.type, stad.QuantitativeDatapoint))
        graph.add((datapoint, stad.hasQualityKind, ex.ObservedTemperature))
        graph.add((datapoint, ex.measuredAt, STATIONS[int(row["Site Id"])]))
        clock = datetime.datetime.strptime(str(row["Time"]), "%H:%M").strftime("%H:%M:%S")
        graph.add((datapoint, ex.observedDate,
                   Literal(stamp(as_date(row["Date"]), clock), datatype=XSD.dateTime, normalize=False)))
        add_value(graph, datapoint, row[TEMP_COLUMN], decimals=None)
    return graph

for name, df in frames.items():
    save_graph(create_ontology_observedTemp(df), f"raw_data_transformed/{name}.ttl")


## Site annual mean temperature



In [ ]:
def create_ontology_site_annualMean(df, graph, year):
    year = int(year)
    site = int(df["Site Id"].iloc[0])
    chronological = df.assign(_instant=pd.to_datetime(df["Date"].astype(str) + " " + df["Time"].astype(str))).sort_values("_instant")
    first, last = chronological.iloc[0], chronological.iloc[-1]
    coverage = ex[f"TimeInterval_{year}"]
    add_interval(graph, coverage, stamp(f"{year}-01-01"), stamp(f"{year}-12-31", "23:00:00"), coverage=True)
    period = ex[f"{site}_{year}_AggregationPeriod"]
    add_convex_period(graph, period, ex[f"{site}_{year}_AggregationPeriodExtent"],
                      stamp(as_date(first["Date"]), datetime.datetime.strptime(str(first["Time"]), "%H:%M").strftime("%H:%M:%S")),
                      stamp(as_date(last["Date"]), datetime.datetime.strptime(str(last["Time"]), "%H:%M").strftime("%H:%M:%S")))
    dataset = ex[f"Dataset_{site}_{year}"]
    add_dataset(graph, dataset, (observation_iri(row) for _, row in df.iterrows()),
                ex[f"{site}_{year}_DatasetDescription"])
    return add_aggregate(graph, ex[f"AnnualMeanTemp_{site}_{year}"], df[TEMP_COLUMN].mean(),
                         ex.AnnualMeanTemperature, ex[f"ArithmeticMeanExecutionAnnualMean_{site}_{year}"],
                         dataset, coverage, period, site_id=site)

graph = new_graph()
annualMeanDict = {"quantity_name": [], "quantity_value": []}
for name, df in frames.items():
    year = int(name.split("=")[-1])
    create_ontology_site_annualMean(df, graph, year)
    if 2013 <= year <= 2022:
        annualMeanDict["quantity_name"].append(f"AnnualMeanTemp_{int(df['Site Id'].iloc[0])}_{year}")
        annualMeanDict["quantity_value"].append(df[TEMP_COLUMN].mean())
save_graph(graph, "annualmeantemp_site.ttl")


## State annual mean temperature

The input dataset consists of the two site annual mean datapoints, so its temporal resolution is one year. Calculation uses unrounded station means.

In [ ]:
def create_ontology_state_annualMean(graph, dic, year):
    year = int(year)
    coverage = ex[f"TimeInterval_{year}"]
    beginning, ending = stamp(f"{year}-01-01"), stamp(f"{year}-12-31", "23:00:00")
    add_interval(graph, coverage, beginning, ending, coverage=True)
    period = ex[f"NH_MeanTemp{year}_AggregationPeriod"]
    add_convex_period(graph, period, ex[f"NH_MeanTemp{year}_AggregationPeriodExtent"], beginning, ending)
    dataset = ex[f"NH_Dataset_{year}"]
    members = [ex[name if name.startswith("AnnualMeanTemp_") else "AnnualMeanTemp_"+name] for name in dic["quantity_name"]]
    add_dataset(graph, dataset, members, ex.AnnualSiteMeans_DatasetDescription, time.unitYear)
    return add_aggregate(graph, ex[f"NH_MeanTemp{year}"], mean(dic["quantity_value"]), ex.AnnualMeanTemperature,
                         ex[f"ArithmeticMeanExecutionAnnual_NH_{year}"], dataset, coverage, period, spatial=True)

graph = new_graph()
for year in range(2013, 2023):
    selected = [(name, df) for name, df in frames.items() if int(name.split("=")[-1]) == year]
    dic = {"quantity_name": [f"AnnualMeanTemp_{int(df['Site Id'].iloc[0])}_{year}" for _, df in selected],
           "quantity_value": [df[TEMP_COLUMN].mean() for _, df in selected]}
    create_ontology_state_annualMean(graph, dic, year)
save_graph(graph, "NH_state_annualMeanTemp.ttl")


## State ten-year annual mean temperature



In [ ]:
def create_ontology_state_10yearMean(graph, dic):
    beginning, ending = stamp("2013-01-01"), stamp("2022-12-31", "23:00:00")
    period = ex["NH_MeanTemp2013-2022_AggregationPeriod"]
    add_convex_period(graph, period, ex["NH_MeanTemp2013-2022_AggregationPeriodExtent"], beginning, ending)
    dataset = ex["Dataset_2013-2022"]
    members = [ex[name if name.startswith("AnnualMeanTemp_") else "AnnualMeanTemp_"+name] for name in dic["quantity_name"]]
    add_dataset(graph, dataset, members, ex.AnnualSiteMeans_DatasetDescription, time.unitYear)
    return add_aggregate(graph, ex["NH_MeanTemp2013-2022"], mean(dic["quantity_value"]), ex.AnnualMeanTemperature,
                         ex["ArithmeticMeanExecution_NH_2013-2022"], dataset, ex["TimeInterval_2013-2022"], period, spatial=True)

graph = new_graph()
create_ontology_state_10yearMean(graph, annualMeanDict)
save_graph(graph, "NH_10yr_meantemp_2013-2022.ttl")


## State annual summer and winter means



In [ ]:
def create_seasonal_mean(df, graph, year, season):
    year = int(year)
    if season == "Summer":
        beginning, ending = stamp(f"{year}-06-01"), stamp(f"{year}-08-31", "23:00:00")
        dataset = ex[f"DatasetNH_Summer_{year}"]
        execution = ex[f"SummerArithmeticMeanExecution_NH_{year}"]
        quality = ex.SummerMeanTemperature
    else:
        last_day = calendar.monthrange(year, 2)[1]
        beginning, ending = stamp(f"{year-1}-12-01"), stamp(f"{year}-02-{last_day:02d}", "23:00:00")
        dataset = ex[f"DatasetWinter_NH_{year}"]
        execution = ex[f"WinterArithmeticMeanExecution_NH_{year}"]
        quality = ex.WinterMeanTemperature
    coverage = ex[f"{season}TimeInterval_{year}"]
    add_interval(graph, coverage, beginning, ending, coverage=True)
    period = ex[f"NH_{season}MeanTemp{year}_AggregationPeriod"]
    add_convex_period(graph, period, ex[f"NH_{season}MeanTemp{year}_AggregationPeriodExtent"], beginning, ending)
    add_dataset(graph, dataset, (observation_iri(row) for _, row in df.iterrows()), ex.SCAN_BaseDataDescription)
    return add_aggregate(graph, ex[f"NH_{season}MeanTemp{year}"], df[TEMP_COLUMN].mean(), quality,
                         execution, dataset, coverage, period, spatial=True)

def create_ontology_state_annualSummerMean(df, graph, year):
    return create_seasonal_mean(df, graph, year, "Summer")

def create_ontology_state_annualWinterMean(df, graph, year):
    return create_seasonal_mean(df, graph, year, "Winter")

annualSummerMeanDict = {"quantity_name": [], "quantity_value": []}
graph = new_graph()
for year in range(2013, 2023):
    df = summer_frame(year)
    create_ontology_state_annualSummerMean(df, graph, year)
    annualSummerMeanDict["quantity_name"].append(f"NH_SummerMeanTemp{year}")
    annualSummerMeanDict["quantity_value"].append(df[TEMP_COLUMN].mean())
save_graph(graph, "NH_state_annualSummerMeanTemp.ttl")
graph = new_graph()
for year in range(2013, 2023):
    create_ontology_state_annualWinterMean(annual_winter_frame(year), graph, year)
save_graph(graph, "NH_state_annualWinterMeanTemp.ttl")


## State ten-year seasonal means

In [ ]:
def create_ontology_state_10yearSummerMean(graph, dic):
    dataset = ex["DatasetSummer_NH_2013-2022"]
    add_dataset(graph, dataset, (ex[name] for name in dic["quantity_name"]),
                ex.AnnualSummerMeans_DatasetDescription, time.unitYear)
    return add_aggregate(graph, ex["NH_SummerMeanTemp2013-2022"], mean(dic["quantity_value"]),
                         ex.SummerMeanTemperature, ex["ArithmeticMeanExecutionSummer_2013-2022"],
                         dataset, ex["TimeInterval_2013-2022"], ex["SummerAggregate2013-2022"], spatial=True)

def create_ontology_state_10yearWinterMean(df, graph):
    dataset = ex["DatasetWinter_NH_2013-2022"]
    add_dataset(graph, dataset, (observation_iri(row) for _, row in df.iterrows()), ex.SCAN_BaseDataDescription)
    return add_aggregate(graph, ex["NH_WinterMeanTemp2013-2022"], df[TEMP_COLUMN].mean(),
                         ex.WinterMeanTemperature, ex["ArithmeticMeanExecutionWinter_2013-2022"],
                         dataset, ex["TimeInterval_2013-2022"], ex["WinterAggregate2013-2022"], spatial=True)

graph = new_graph()
create_ontology_state_10yearSummerMean(graph, annualSummerMeanDict)
save_graph(graph, "NH_state_10yearSummerMeanTemp.ttl")
# Preserve the original ten-year winter selection: Jan-Feb and Dec of each calendar year.
winter_df = pd.concat([year_frame(year).loc[lambda df: df["Date"].dt.month.isin([1, 2, 12])]
                      for year in range(2013, 2023)], ignore_index=True)
graph = new_graph()
create_ontology_state_10yearWinterMean(winter_df, graph)
save_graph(graph, "NH_state_10-yearWinterMeanTemp.ttl")


## Daily aggregate builders

In [ ]:
# Respect explicitly added H/MM examples in the updated manual sample.
SPECIAL_MEAN_CASES = {}
site_mean_fixture_path = SAMPLE_DIR / "Dailymeantemp_site.ttl"
if site_mean_fixture_path.is_file():
    fixture = read_sample_graph(site_mean_fixture_path)
    for datapoint in fixture.subjects(stad.hasQualityKind, ex.DailyMeanTemperature):
        match = re.fullmatch(re.escape(str(ex)) + r"DailyMeanTemperature_(\d+)_(\d{8})_(H|MM)", str(datapoint))
        if match:
            site, day, variant = match.groups()
            SPECIAL_MEAN_CASES.setdefault((site, day), set()).add(variant)
    del fixture
    print(f"Loaded {len(SPECIAL_MEAN_CASES)} alternate daily-mean examples from the sample fixture.")

DAILY = {
    "mean": ("DailyMeanTemperature", ex.DailyMeanTemperature, "ArithmeticMeanExecution"),
    "min": ("DailyMinTemperature", ex.DailyMinimumTemperature, "MinimumCalculationExecution"),
    "max": ("DailyMaxTemperature", ex.DailyMaximumTemperature, "MaximumCalculationExecution"),
}

def create_daily_graph(df, graph, method, site_id=None, base_df=None):
    if base_df is None:
        raise ValueError("Pass the hourly base_df so daily dataset membership can be recorded")
    name, quality, execution_prefix = DAILY[method]
    grouped = {as_date(day): rows for day, rows in base_df.groupby("Date")}
    for _, row in df.iterrows():
        day = as_date(row["Date"])
        day_id = day.strftime("%Y%m%d")
        location = str(int(site_id)) if site_id is not None else "NH"
        coverage = ex[f"{location}_{day_id}_TemporalCoverage"]
        beginning, ending = stamp(day), stamp(day, "23:00:00")
        add_interval(graph, coverage, beginning, ending, coverage=True)
        period = ex[f"{location}_{day_id}_AggregationPeriod"]
        add_convex_period(graph, period, ex[f"{location}_{day_id}_AggregationPeriodExtent"], beginning, ending)
        dataset = ex[f"{location}_{day_id}_Dataset"] if site_id is not None else ex[f"Dataset_NH_{day_id}"]
        description = ex[f"{location}_{day_id}_DatasetDescription"] if site_id is not None else ex.SCAN_BaseDataDescription
        add_dataset(graph, dataset, (observation_iri(item) for _, item in grouped[day].iterrows()), description)
        # The manual state's daily mean execution has DMT in its identifier.
        execution = (ex[f"ArithmeticMeanExecutionDMT_NH_{day_id}"] if site_id is None and method == "mean"
                     else ex[f"{execution_prefix}_{location}_{day_id}"])
        variants = SPECIAL_MEAN_CASES.get((location, day_id), set()) if method == "mean" else set()
        if site_id is None and method == "mean":
            datapoint = ex[f"SCAN_{name}_{location}_{day_id}"]
        elif "H" in variants:
            datapoint = ex[f"{name}_{location}_{day_id}_H"]
            dataset = ex[f"Hourly_{location}_{day_id}_Dataset"]
            add_dataset(graph, dataset, (observation_iri(item) for _, item in grouped[day].iterrows()), description)
        else:
            datapoint = ex[f"{name}_{location}_{day_id}"]
        add_aggregate(graph, datapoint, row[TEMP_COLUMN], quality, execution, dataset, coverage,
                      period, method=method, site_id=site_id, spatial=site_id is None)
        if site_id is not None and method == "mean":
            graph.add((datapoint, stad.hasSpatialCoverage, STATIONS[int(site_id)]))
        if "H" in variants:
            graph.add((datapoint, RDFS.label, Literal("Mean temperature of hourly readings at " +
                                                       str(STATIONS[int(site_id)]).removeprefix(str(ex)))))
        if "MM" in variants:
            # This is a distinct execution: its input is the two daily extrema, not hourly rows.
            mm_datapoint = ex[f"{name}_{location}_{day_id}_MM"]
            mm_dataset = ex[f"Day_MinMax_{location}_{day_id}_Dataset"]
            members = [ex[f"DailyMinTemperature_{location}_{day_id}"],
                       ex[f"DailyMaxTemperature_{location}_{day_id}"]]
            add_dataset(graph, mm_dataset, members, temporal_unit=time.unitDay)
            mm_value = pd.Series([grouped[day][TEMP_COLUMN].min(), grouped[day][TEMP_COLUMN].max()]).mean()
            add_aggregate(graph, mm_datapoint, mm_value, quality,
                          ex[f"ArithmeticMeanExecution_{location}_{day_id}_MM"], mm_dataset,
                          coverage, period, method="mean", site_id=site_id)
            graph.add((mm_datapoint, stad.hasSpatialCoverage, STATIONS[int(site_id)]))
            graph.add((mm_datapoint, RDFS.label, Literal("Mean temperature of daily min and max at " +
                                                         str(STATIONS[int(site_id)]).removeprefix(str(ex)))))
    return graph

def create_ontology_site_DailyMean(df, graph, site_id, base_df=None):
    return create_daily_graph(df, graph, "mean", site_id, base_df)

def create_ontology_site_DailyMin(df, graph, site_id, base_df=None):
    return create_daily_graph(df, graph, "min", site_id, base_df)

def create_ontology_site_DailyMax(df, graph, site_id, base_df=None):
    return create_daily_graph(df, graph, "max", site_id, base_df)

def create_ontology_state_DailyMean(df, graph, base_df=None):
    return create_daily_graph(df, graph, "mean", base_df=base_df)

def create_ontology_state_DailyMin(df, graph, base_df=None):
    return create_daily_graph(df, graph, "min", base_df=base_df)

def create_ontology_state_DailyMax(df, graph, base_df=None):
    return create_daily_graph(df, graph, "max", base_df=base_df)


## Site daily mean, minimum, and maximum temperatures



In [ ]:
for method, filename in [("mean", "Dailymeantemp_site.ttl"), ("min", "Dailymintemp_site.ttl"),
                          ("max", "Dailymaxtemp_site.ttl")]:
    graph = new_graph()
    for df in frames.values():
        aggregate_df = df.groupby("Date")[TEMP_COLUMN].agg(method).reset_index()
        create_daily_graph(aggregate_df, graph, method, int(df["Site Id"].iloc[0]), df)
    save_graph(graph, filename)


## State daily mean, minimum, and maximum temperatures



In [ ]:
for method, filename in [("mean", "DailyMeanTemp_NH.ttl"), ("min", "DailyMinTemp_NH.ttl"),
                          ("max", "DailyMaxTemp_NH.ttl")]:
    graph = new_graph()
    for year in range(2012, 2023):
        df = year_frame(year)
        aggregate_df = df.groupby("Date")[TEMP_COLUMN].agg(method).reset_index()
        create_daily_graph(aggregate_df, graph, method, base_df=df)
    save_graph(graph, filename)


## County and optional state geometry

In [ ]:
def create_ontology_NH_Counties(df, graph):
    for _, row in df.iterrows():
        feature = ex[str(row["county_nam"])]
        geometry = stable_blank(feature, "geometry")
        graph.add((feature, RDF.type, geo.Feature))
        graph.add((feature, RDFS.label, Literal(str(row["county_nam"]), datatype=XSD.string)))
        graph.add((feature, geo.hasGeometry, geometry))
        graph.add((geometry, RDF.type, geo.Geometry))
        graph.add((geometry, geo.asWKT, Literal(
            "<http://www.opengis.net/def/crs/EPSG/8.5/4326>"+str(row["WKT"]), datatype=geo.wktLiteral)))
    return graph

graph = new_graph()
create_ontology_NH_Counties(pd.read_csv(DATA_DIR / "NH_counties.csv"), graph)
save_graph(graph, "NH_Counties.ttl")